In [ ]:

import random
import math
import operator

# ==========================================
# 1. GEP CONFIGURATION
# ==========================================

FUNCTIONS = {
    '+': (operator.add, 2),
    '-': (operator.sub, 2),
    '*': (operator.mul, 2),
    '/': (lambda a, b: a / b if abs(b) > 1e-10 else 1e10, 2)
}

TERMINALS = ['x']

HEAD_LENGTH = 6
MAX_ARITY = 2

TAIL_LENGTH = HEAD_LENGTH * (MAX_ARITY - 1) + 1
GENE_LENGTH = HEAD_LENGTH + TAIL_LENGTH

NUM_GENES = 3
POPULATION_SIZE = 100
GENERATIONS = 100

MUTATION_RATE = 0.02
CROSSOVER_RATE = 0.7
ELITE_SIZE = 2

# ==========================================
# 2. CHROMOSOME GENERATION
# ==========================================

def create_gene():
    head = [
        random.choice(list(FUNCTIONS.keys()) + TERMINALS)
        for _ in range(HEAD_LENGTH)
    ]

    tail = [
        random.choice(TERMINALS)
        for _ in range(TAIL_LENGTH)
    ]

    return head + tail


def create_chromosome():
    return [
        create_gene()
        for _ in range(NUM_GENES)
    ]


def create_population():
    return [
        create_chromosome()
        for _ in range(POPULATION_SIZE)
    ]


# ==========================================
# 3. GENE TO EXPRESSION TREE
# ==========================================

def decode_gene(gene):
    """
    Convert a linear GEP gene into an expression tree.
    """

    if gene[0] not in FUNCTIONS:
        return gene[0]

    expression = []
    positions = [0]

    for symbol in gene:
        if not positions:
            break

        position = positions.pop(0)

        if symbol in FUNCTIONS:
            function, arity = FUNCTIONS[symbol]

            expression.append((symbol, arity))

            for _ in range(arity):
                positions.append(len(expression))

        else:
            expression.append(symbol)

    return expression


# ==========================================
# 4. EVALUATE EXPRESSION TREE
# ==========================================

def evaluate_gene(gene, x):

    if isinstance(gene, str):
        if gene == 'x':
            return x

        return 0

    if not gene:
        return 0

    def build(index):

        if index >= len(gene):
            return 0, index

        node = gene[index]

        if isinstance(node, str):
            if node == 'x':
                return x, index + 1
            return 0, index + 1

        symbol, arity = node

        function = FUNCTIONS[symbol][0]

        arguments = []

        next_index = index + 1

        for _ in range(arity):
            value, next_index = build(next_index)
            arguments.append(value)

        try:
            result = function(*arguments)

            if not math.isfinite(result):
                return 1e10, next_index

            return max(min(result, 1e10), -1e10), next_index

        except (ArithmeticError, ValueError, OverflowError):
            return 1e10, next_index

    return build(0)[0]


# ==========================================
# 5. MULTIGENIC CHROMOSOME EVALUATION
# ==========================================

def evaluate_chromosome(chromosome, x):

    result = 0

    for gene in chromosome:
        tree = decode_gene(gene)
        result += evaluate_gene(tree, x)

    return result


# ==========================================
# 6. FITNESS FUNCTION
# ==========================================

def target_function(x):
    return x**4 + x**3 + x**2 + x


def fitness(chromosome):

    total_error = 0

    fitness_cases = [
        -2, -1, 0, 1, 2, 3, 4, 5
    ]

    for x in fitness_cases:

        predicted = evaluate_chromosome(chromosome, x)
        actual = target_function(x)

        error = abs(predicted - actual)

        if not math.isfinite(error):
            return 0

        total_error += error

    return 1 / (1 + total_error)


# ==========================================
# 7. MUTATION
# ==========================================

def mutate(chromosome):

    new_chromosome = [
        gene.copy() for gene in chromosome
    ]

    for gene in new_chromosome:

        for i in range(len(gene)):

            if random.random() < MUTATION_RATE:

                if i < HEAD_LENGTH:

                    gene[i] = random.choice(
                        list(FUNCTIONS.keys()) + TERMINALS
                    )

                else:

                    gene[i] = random.choice(TERMINALS)

    return new_chromosome


# ==========================================
# 8. ONE-POINT CROSSOVER
# ==========================================

def one_point_crossover(parent1, parent2):

    child1 = [
        gene.copy() for gene in parent1
    ]

    child2 = [
        gene.copy() for gene in parent2
    ]

    flat1 = sum(child1, [])
    flat2 = sum(child2, [])

    point = random.randint(1, len(flat1) - 1)

    new1 = flat1[:point] + flat2[point:]
    new2 = flat2[:point] + flat1[point:]

    child1 = [
        new1[i:i + GENE_LENGTH]
        for i in range(0, len(new1), GENE_LENGTH)
    ]

    child2 = [
        new2[i:i + GENE_LENGTH]
        for i in range(0, len(new2), GENE_LENGTH)
    ]

    return child1, child2


# ==========================================
# 9. TWO-POINT CROSSOVER
# ==========================================

def two_point_crossover(parent1, parent2):

    flat1 = sum([g.copy() for g in parent1], [])
    flat2 = sum([g.copy() for g in parent2], [])

    p1, p2 = sorted(
        random.sample(range(1, len(flat1)), 2)
    )

    child1 = flat1[:p1] + flat2[p1:p2] + flat1[p2:]
    child2 = flat2[:p1] + flat1[p1:p2] + flat2[p2:]

    return (
        [child1[i:i + GENE_LENGTH]
         for i in range(0, len(child1), GENE_LENGTH)],

        [child2[i:i + GENE_LENGTH]
         for i in range(0, len(child2), GENE_LENGTH)]
    )


# ==========================================
# 10. GENE RECOMBINATION
# ==========================================

def gene_recombination(parent1, parent2):

    child1 = [g.copy() for g in parent1]
    child2 = [g.copy() for g in parent2]

    index = random.randrange(NUM_GENES)

    child1[index], child2[index] = (
        child2[index], child1[index]
    )

    return child1, child2


# ==========================================
# 11. SELECTION
# ==========================================

def selection(population):

    scores = [fitness(ind) for ind in population]

    total = sum(scores)

    if total == 0:
        return random.choice(population)

    return random.choices(
        population,
        weights=scores,
        k=1
    )[0]


# ==========================================
# 12. EVOLUTIONARY ALGORITHM
# ==========================================

def run_gep():

    population = create_population()

    best_solution = None
    best_fitness = -1

    for generation in range(GENERATIONS):

        scores = [
            fitness(ind) for ind in population
        ]

        ranked = sorted(
            zip(scores, population),
            key=lambda item: item[0],
            reverse=True
        )

        current_best_fitness, current_best = ranked[0]

        if current_best_fitness > best_fitness:

            best_fitness = current_best_fitness
            best_solution = [
                gene.copy() for gene in current_best
            ]

        print(
            f"Generation {generation + 1}: "
            f"Best Fitness = {best_fitness:.8f}"
        )

        # Elitism
        new_population = [
            [gene.copy() for gene in ind]
            for _, ind in ranked[:ELITE_SIZE]
        ]

        while len(new_population) < POPULATION_SIZE:

            parent1 = selection(population)
            parent2 = selection(population)

            child1 = [
                gene.copy() for gene in parent1
            ]

            child2 = [
                gene.copy() for gene in parent2
            ]

            if random.random() < CROSSOVER_RATE:

                operator_choice = random.choice([
                    "one_point",
                    "two_point",
                    "gene"
                ])

                if operator_choice == "one_point":

                    child1, child2 = one_point_crossover(
                        parent1, parent2
                    )

                elif operator_choice == "two_point":

                    child1, child2 = two_point_crossover(
                        parent1, parent2
                    )

                else:

                    child1, child2 = gene_recombination(
                        parent1, parent2
                    )

            child1 = mutate(child1)
            child2 = mutate(child2)

            new_population.append(child1)

            if len(new_population) < POPULATION_SIZE:
                new_population.append(child2)

        population = new_population

    return best_solution, best_fitness


# ==========================================
# 13. RUN GEP
# ==========================================

if __name__ == "__main__":

    best, score = run_gep()

    print("\nBest Fitness:", score)

    print("\nBest Chromosome:")

    for i, gene in enumerate(best):
        print(f"Gene {i + 1}: {''.join(gene)}")

    print("\nPredictions:")

    for x in [1, 2, 3, 4]:

        predicted = evaluate_chromosome(best, x)
        actual = target_function(x)

        print(
            f"x={x}, "
            f"Predicted={predicted}, "
            f"Actual={actual}"
        )


Generation 1: Best Fitness = 0.00460829
Generation 2: Best Fitness = 0.00460829
Generation 3: Best Fitness = 0.00591716
Generation 4: Best Fitness = 0.00591716
Generation 5: Best Fitness = 0.00591716
Generation 6: Best Fitness = 0.01369863
